In [4]:
a = '20.1'
a.isfloat()

AttributeError: 'str' object has no attribute 'isfloat'

In [ ]:
!pip install trl
!pip install -U bitsandbytes
!pip install latex2sympy2
!pip install unsloth vllm
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 19.5 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0


In [ ]:
!pip uninstall -y trl
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install git+https://github.com/huggingface/trl.git
!pip install vllm

Found existing installation: trl 0.8.6
Uninstalling trl-0.8.6:
  Successfully uninstalled trl-0.8.6
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-bapjesl8/unsloth_fed55734e6f34285bc3368052a8160c8
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-bapjesl8/unsloth_fed55734e6f34285bc3368052a8160c8
  Resolved https://github.com/unslothai/unsloth.git to commit 521e20127dac99a845abc78cb95d13040ace6616
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Cloning https://github.com/huggingface/trl.git to /tmp/pip-req-build-u0u61kme
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/trl.git /tmp/pip-req-build-u0u61kme
  Resolved https://github.com/huggingface/trl.git to commit fda5a7

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
from unsloth import is_bfloat16_supported
from datasets import load_dataset, Dataset
import pandas as pd
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from latex2sympy2 import latex2sympy
import re


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find Trainer class in trl.trainer.bco_trainer. Found: ['BCOTrainer', '_BCOTrainer']
Unsloth: Could not find Trainer class in trl.trainer.cpo_trainer. Found: ['CPOTrainer', '_CPOTrainer']
Unsloth: Could not find Trainer class in trl.trainer.gkd_trainer. Found: ['GKDTrainer', '_GKDTrainer']
Unsloth: Could not find Trainer class in trl.trainer.kto_trainer. Found: ['KTOTrainer', '_KTOTrainer']
Unsloth: Could not find Trainer class in trl.trainer.nash_md_trainer. Found: ['NashMDTrainer', '_NashMDTrainer']
Unsloth: Could not find Trainer class in trl.trainer.online_dpo_trainer. Found: ['OnlineDPOTrainer', '_OnlineDPOTrainer']
Unsloth: Could not find Trainer class in trl.trainer.orpo_trainer. Found: ['ORPOTrainer', '_ORPOTrainer']
Unsloth: Could not find Trainer class in trl.trainer.ppo_trainer. Found: ['PPOTrainer', '_PPOTrainer']
Unslot

In [ ]:
ds = load_dataset("AI-MO/NuminaMath-CoT")
train_ds = ds["train"]
test = ds["test"]


test_df = test.to_pandas()
# included ['cn_k12', 'orca_math']
problematic_sources = ['olympiads', 'aops_forum', 'synthetic_math',  'amc_aime', 'synthetic_amc', 'math', 'gms8k']
test_df = test_df[~test_df['source'].isin(problematic_sources)]

test_ds = Dataset.from_pandas(test_df)

train_df = train_ds.to_pandas()
problematic_sources = ['olympiads', 'aops_forum', 'cn_k12', 'orca_math', 'synthetic_math',  'amc_aime', 'synthetic_amc']
train_df = train_df[~train_df['source'].isin(problematic_sources)]

train_ds = Dataset.from_pandas(train_df)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00003-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00004-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/166k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/859494 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
train_df.head()

,source,problem,solution,messages
89,math,Find all real numbers $x$ so that the product ...,To find all real numbers $x$ such that the pro...,[{'content': 'Find all real numbers $x$ so tha...
123,gsm8k,Every time Carl earned $0.50 he would go to th...,To determine how many candy bars Carl can buy ...,[{'content': 'Every time Carl earned $0.50 he ...
175,gsm8k,Ruth goes to school 8 hours a day and 5 days a...,"To solve this problem, we start by calculating...",[{'content': 'Ruth goes to school 8 hours a da...
197,gsm8k,There are twice as many cows in Devonshire as ...,"To solve this problem, let's break it down int...",[{'content': 'There are twice as many cows in ...
228,gsm8k,"Tom and Tim both brought 4, six-sided dice to ...","To solve the problem, we start by calculating ...","[{'content': 'Tom and Tim both brought 4, six-..."


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#gets exact answer from boxed{}
def extract_xml_answer(text: str) -> str:
    answer = text.split("\\boxed{")[-1]
    answer = answer.split("}")[0]
    return answer



In [ ]:
#puts prompt into the dataset, so that LLM knows they are responding to a question instead of memorizing conversations
def process_data_for_grpo(example):
    sys_prompt = "Please reason step by step, and put your final answer within \\boxed{} and put units of measurement where appropriate."
    raw_problem = example['problem']

    messages = [
        {
            "role": "user",
            "content": f"{sys_prompt}\n\n{raw_problem}"
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    ground_truth = example['solution']
    print(ground_truth)
    ground_truth = extract_xml_answer(ground_truth)
    print(ground_truth)

    return {
        "prompt": prompt_text,
        "answer": ground_truth
    }


def reward_function(prompts, completions, answer, **kwargs):
    responses = []



    for completion, truth in zip(completions, answer):
        score = 0.0


        #getting the ground truth
        truth_str = str(truth)
        if "####" in truth_str:
            truth_clean = truth_str.split("####")[-1].strip()
        else:
            truth_clean = truth_str.strip()


        # +0.1 if the response is over 200 characters to encourage thinking
        if len(completion) > 200:
            score += 0.1


        box_match = re.search(r"\\boxed\{(.*?)\}", completion)
        predicted_content = None

        # +0.1 if the response is in the boxed{} syntax
        if box_match:
            score += 0.1
            predicted_content = box_match.group(1).strip()

        #check to see if the prediction is correct
        if predicted_content:
            if predicted_content == truth_clean:
                score += 1.0
            else:
                try:
                    if latex2sympy(predicted_content).equals(latex2sympy(truth_clean)):
                        score += 1.0
                except:
                    pass

        #0.5 if the correct answer is in the prompt
        if score < 0.5:
            if truth_clean in completion:
                score += 0.5

        responses.append(score)

    return responses

In [ ]:
import sys
import os
from unsloth import FastLanguageModel, PatchFastRL
from unsloth import is_bfloat16_supported
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer
import torch
from peft import PeftModel

if not hasattr(sys.stdout, 'fileno'):
    sys.stdout.fileno = lambda: 1
else:
    # Sometimes Colab resets it, so we force it just in case
    try:
        sys.stdout.fileno()
    except:
        sys.stdout.fileno = lambda: 1

if not hasattr(sys.stderr, 'fileno'):
    sys.stderr.fileno = lambda: 2
else:
    try:
        sys.stderr.fileno()
    except:
        sys.stderr.fileno = lambda: 2

print("Colab 'fileno' patch applied successfully.")

PatchFastRL("GRPO", FastLanguageModel)

# ==========================================
# 2. LOAD TOKENIZER & RESIZE MODEL
# ==========================================
ft_path = "/content/drive/MyDrive/CIS 602 Project 2/fine-tuning/SFT4"
base_model_name = "mistralai/Mistral-7B-Instruct-v0.2"
merged_save_path = "./sft_merged_model"

# Only run merge if we haven't done it yet
if not os.path.exists(merged_save_path):
    print("⚡ Phase 1: Merging SFT Adapter to fix Vocab Mismatch...")

    # A. Load Tokenizer to get size
    tokenizer_sft = AutoTokenizer.from_pretrained(ft_path)
    vocab_size = len(tokenizer_sft)

    # B. Load Base Model (CPU/RAM efficient load for merging)
    model_temp, _ = FastLanguageModel.from_pretrained(
        model_name = base_model_name,
        load_in_4bit = False, # Load in 16bit for merging accuracy
        gpu_memory_utilization = 0.6,
    )

    print("   Resizing embeddings...")
    model_temp.resize_token_embeddings(vocab_size)

    # D. Force-Wrap in PeftModel (The "Hammer")
    # This explicitly tells Python: "This is a LoRA model, load these weights."
    print("   Loading adapter weights...")
    model_temp = PeftModel.from_pretrained(model_temp, ft_path)

    # E. Merge
    print("   Merging weights...")
    model_temp = model_temp.merge_and_unload()

    # F. Save
    print(f"   Saving to {merged_save_path}...")
    model_temp.save_pretrained(merged_save_path)
    tokenizer_sft.save_pretrained(merged_save_path)

    # Cleanup
    del model_temp, tokenizer_sft
    torch.cuda.empty_cache()
    print("   Merge complete. VRAM cleared.")
else:
    print("⚡ Phase 1: Merged model found. Skipping.")

# ==========================================
# PHASE 2: LOAD FOR RL (Fast Mode)
# ==========================================
print("⚡ Phase 2: Loading Merged Model for RL...")

# A. Load the CLEAN merged model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = merged_save_path, # Load the local merged model
    max_seq_length = 2048,
    load_in_4bit = True,           # Now we go back to 4-bit for speed
    fast_inference = True,         # vLLM will work now!
    gpu_memory_utilization = 0.5,
)

# B. Add a FRESH LoRA Adapter for the RL stage
# We need new trainable weights for the RL to update
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Dropout = 0 is standard for Unsloth
    bias = "none",
    use_gradient_checkpointing = False,
    random_state = 3407,
)

dataset = train_ds.map(
    process_data_for_grpo,
    remove_columns=train_ds.column_names
)

grpo_dataset = dataset.shuffle(seed=42).select(range(800))

# ==========================================
# 3. CONFIG & TRAIN
# ==========================================
training_args = GRPOConfig(
    output_dir = "grpo_unsloth_final",
    learning_rate = 5e-5,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 2,
    num_generations = 8,
    max_prompt_length = 256,
    max_completion_length = 2048,
    num_train_epochs = 1,
    report_to = "wandb",
    use_vllm = True,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    beta = 0.01,
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = reward_function, # Make sure this is defined!
    args = training_args,
    train_dataset = grpo_dataset,   # Make sure this is defined!
    tokenizer = tokenizer,
)

model.print_trainable_parameters()

print("Starting Unsloth GRPO (Mismatch Fixed)...")
trainer.train()
trainer.save_model("rl_math_model_final")

Colab 'fileno' patch applied successfully.
Unsloth: Could not find Trainer class in trl.trainer.bco_trainer. Found: ['BCOTrainer', '_BCOTrainer']
Unsloth: Could not find Trainer class in trl.trainer.cpo_trainer. Found: ['CPOTrainer', '_CPOTrainer']
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: Could not find Trainer class in trl.trainer.gkd_trainer. Found: ['GKDTrainer', '_GKDTrainer']
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: Could not find Trainer class in trl.trainer.kto_trainer. Found: ['KTOTrainer', '_KTOTrainer']
Unsloth: Could not find Trainer class in trl.trainer.nash_md_trainer. Found: ['NashMDTrainer', '_NashMDTrainer']
Unsloth: Could not find Trainer class in trl.trainer.online_dpo_trainer. Found: ['OnlineDPOTrainer', '_OnlineDPOTrainer']
Unsloth: Could not find Trainer class in trl.trainer.orpo_trainer. Found: ['ORPOTrainer', '_ORPOTrainer']
Unsloth: Could not find Trainer class in trl.trainer.ppo_trainer. Found: ['PPOTrainer', '_PPOTrainer']
Unslot

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

...

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: root-aidan (mahnoorkhalid156-university-of-massachusetts) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


WARNING 12-08 20:37:12 [input_processor.py:243] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!


UnboundLocalError: cannot access local variable 'tracer_output' where it is not associated with a value

In [ ]:
# 1. Mount Drive (if not already mounted)
from google.colab import drive
drive.mount('/content/drive')

# 2. Define a path in your Drive
# Make sure this folder exists or let the code create it
drive_save_path = "/content/drive/MyDrive/CIS 602 Project 2/mix_matched"

# 3. Save the Model (The LoRA Adapters)
print(f"Saving model to {drive_save_path}...")
trainer.save_model(drive_save_path)

# 4. Save the Tokenizer (Critical for reloading)
tokenizer.save_pretrained(drive_save_path)

print("✅ Model saved to Google Drive. You can now safely close Colab.")

In [ ]:
# ==============================================================================
# 🏁 PHASE 4: FINAL EVALUATION (VOCAB FIX + SAFE INFERENCE)
# ==============================================================================
import gc
import torch
import pandas as pd
import re
import json
import os
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from latex2sympy2 import latex2sympy
from tqdm import tqdm

print("\n\n🛑 TRAINING FINISHED. STARTING EVALUATION...")

# 1. CLEANUP MEMORY
try:
    del model
    del tokenizer
    del trainer
except:
    pass
torch.cuda.empty_cache()
gc.collect()
print("🧹 VRAM cleared.")

# ==========================================
# 0. CONFIGURATION & HOTFIX
# ==========================================
NUM_SAMPLES = 5
SFT_BASE_PATH = "./sft_merged_model"
RL_ADAPTER_PATH = "/content/drive/MyDrive/CIS 602 Project 2/mix_matched"

# --- 🛠️ THE HOTFIX: Patch the config.json ---
print(f"🛠️ Patching config.json in {SFT_BASE_PATH} to fix vocab size...")
config_path = os.path.join(SFT_BASE_PATH, "config.json")

if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = json.load(f)

    # Force vocab size to match the weights (32001)
    if config.get("vocab_size") != 32001:
        print(f"   ⚠️ Found vocab_size={config.get('vocab_size')}. Updating to 32001.")
        config["vocab_size"] = 32001

        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)
        print("   ✅ Config patched.")
    else:
        print("   ✅ Config already correct.")
else:
    print("   ⚠️ Config file not found, skipping patch.")

# ==========================================
# 1. DATASET SETUP
# ==========================================
print("📂 Setting up Dataset...")
ds = load_dataset("AI-MO/NuminaMath-CoT")
test = ds["test"]

test_df = test.to_pandas()
problematic_sources = ['olympiads', 'aops_forum', 'synthetic_math', 'amc_aime', 'synthetic_amc', 'math', 'gms8k']
test_df = test_df[~test_df['source'].isin(problematic_sources)]
test_ds = Dataset.from_pandas(test_df)

# ==========================================
# 2. REWARD FUNCTION
# ==========================================
def reward_function(prompts, completions, answer, **kwargs):
    responses = []
    for completion, truth in zip(completions, answer):
        score = 0.0

        truth_str = str(truth)
        if "####" in truth_str:
            truth_clean = truth_str.split("####")[-1].strip()
        else:
            truth_clean = truth_str.strip()

        if len(completion) > 200:
            score += 0.1

        box_match = re.search(r"\\boxed\{(.*?)\}", completion)
        predicted_content = None
        if box_match:
            score += 0.1
            predicted_content = box_match.group(1).strip()

        found_correct = False
        if truth_clean in completion:
            found_correct = True

        if not found_correct and predicted_content:
            try:
                if latex2sympy(predicted_content).equals(latex2sympy(truth_clean)):
                    found_correct = True
            except:
                pass

        if found_correct:
            score += 1.0

        responses.append(score)
    return responses

# ==========================================
# 3. PREPARE PROMPTS
# ==========================================
eval_samples = test_ds.select(range(NUM_SAMPLES))
prompts = []
ground_truths = []
sys_prompt = "Please reason step by step, and put your final answer within \\boxed{} and put units of measurement where appropriate."

print("⏳ Preparing Prompts...")
# We load just for tokenization first
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_BASE_PATH,
    load_in_4bit = True,
    # IMPORTANT: fast_inference=False to avoid vLLM crashes on vocab issues
    fast_inference = False
)

for item in eval_samples:
    raw_problem = item['problem']
    messages = [{"role": "user", "content": f"{sys_prompt}\n\n{raw_problem}"}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(formatted_prompt)

    truth = item['solution']
    if "####" in truth: truth = truth.split("####")[-1].strip()
    ground_truths.append(truth.strip())

# ==========================================
# 4. RUN GENERATION (SAFE MODE)
# ==========================================
FastLanguageModel.for_inference(model)

# --- A. GENERATE SFT RESPONSES ---
print("🚀 Generating SFT Responses (Standard Inference)...")
inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")

with torch.no_grad():
    outputs_sft = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
sft_completions = tokenizer.batch_decode(outputs_sft[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"   Sample SFT: {sft_completions[0][:200]}...")

# --- B. ATTACH RL ADAPTER ---
print(f"\n🔗 Attaching RL Adapter from: {RL_ADAPTER_PATH}...")
model.load_adapter(RL_ADAPTER_PATH)

# --- C. GENERATE RL RESPONSES ---
print("🚀 Generating RL Responses (Standard Inference)...")
with torch.no_grad():
    outputs_rl = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
rl_completions = tokenizer.batch_decode(outputs_rl[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"   Sample RL: {rl_completions[0][:200]}...")

# ==========================================
# 5. SCORING & SAVING
# ==========================================
print("\n⚖️ Calculating Rewards...")
sft_scores = reward_function(prompts, sft_completions, ground_truths)
rl_scores = reward_function(prompts, rl_completions, ground_truths)

df = pd.DataFrame({
    'Prompt': prompts,
    'Ground Truth': ground_truths,
    'SFT Response': sft_completions,
    'SFT Score': sft_scores,
    'RL Response': rl_completions,
    'RL Score': rl_scores
})

df['Winner'] = df.apply(lambda x: 'RL' if x['RL Score'] > x['SFT Score'] else ('SFT' if x['SFT Score'] > x['RL Score'] else 'Tie'), axis=1)

print("\n" + "="*40)
print(f"🏁 FINAL RESULTS")
print(f"Avg SFT Score: {sum(sft_scores)/len(sft_scores):.4f}")
print(f"Avg RL Score:  {sum(rl_scores)/len(rl_scores):.4f}")
print(f"RL Wins:  {len(df[df['Winner']=='RL'])}")
print(f"SFT Wins: {len(df[df['Winner']=='SFT'])}")
print("="*40)

save_path = "/content/drive/MyDrive/CIS 602 Project 2/final_safe_comparison.csv"
df.to_csv(save_path, index=False)
print(f"✅ Saved results to {save_path}")



🛑 TRAINING FINISHED. STARTING EVALUATION...


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import os
import shutil

# Define source and destination paths
model_source_path = "./SFT/MistralAI-7B"
drive_destination_path = "/content/drive/MyDrive/CIS 602 Project 2/RL_HF/Diogenes"

# Check if the source directory exists
if not os.path.exists(model_source_path):
    print(f"Error: Source directory '{model_source_path}' does not exist.")
elif os.path.exists(drive_destination_path):
    print(f"Destination directory '{drive_destination_path}' already exists. Skipping copy.")
    print("If you wish to overwrite, please delete the existing directory in your Google Drive and rerun this cell.")
else:
    # Create parent directories if they don't exist
    os.makedirs(os.path.dirname(drive_destination_path), exist_ok=True)

    # Copy the directory
    try:
        shutil.copytree(model_source_path, drive_destination_path)
        print(f"Successfully copied '{model_source_path}' to '{drive_destination_path}'")
    except Exception as e:
        print(f"An error occurred during copying: {e}")